# Open a CEFI Icechunk dataset

<br>
You will need the packages `icechunk` and `xarray`.

## Set up the config and access

In [2]:
import icechunk
import xarray as xr

# -------------------------------------------------------------------
# 1. Source data: original NOAA CEFI NetCDF files
# -------------------------------------------------------------------
source_data_bucket = "s3://noaa-oar-cefi-regional-mom6-pds"
source_data_prefix = "northeast_pacific/full_domain/hindcast/daily/regrid/r20250912"
source_data_region = "us-east-1"

source_data_url_prefix = source_data_bucket.rstrip("/") + "/"

# -------------------------------------------------------------------
# 2. Icechunk repo: public Source Cooperative location
# -------------------------------------------------------------------
icechunk_bucket = "us-west-2.opendata.source.coop"
icechunk_prefix = "eeholmes/cefi/nepacific-icechunk"
icechunk_region = "us-west-2"

# -------------------------------------------------------------------
# 3. Authorize access to the original NOAA virtual chunks
# -------------------------------------------------------------------
# The Icechunk repo contains virtual references back to the public
# NOAA CEFI S3 bucket. Use anonymous access for these chunks.
credentials = icechunk.containers_credentials({
    source_data_url_prefix: icechunk.s3_credentials(anonymous=True)
})

# -------------------------------------------------------------------
# 4. Tell Icechunk which external virtual chunk locations are allowed
# -------------------------------------------------------------------
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=source_data_url_prefix,
        store=icechunk.s3_store(
            region=source_data_region,
            anonymous=True,
        ),
    ),
)

# -------------------------------------------------------------------
# 5. Point to the public Icechunk repo on Source Cooperative
# -------------------------------------------------------------------
storage = icechunk.s3_storage(
    bucket=icechunk_bucket,
    prefix=icechunk_prefix,
    region=icechunk_region,
    anonymous=True,
)

# -------------------------------------------------------------------
# 6. Open the Icechunk repo
# -------------------------------------------------------------------
repo = icechunk.Repository.open(
    storage,
    config=config,
    authorize_virtual_chunk_access=credentials,
)

session = repo.readonly_session(branch="main")

In [5]:
# Turn off warning
import warnings
warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

## Open a group

* `monthly/main`
* `monthly/aux`

In [6]:
ds = xr.open_zarr(
    session.store,
    group="monthly/main",
    consolidated=False,
    chunks=None,
)

ds

<xarray.Dataset> Size: 2TB
Dimensions:                      (time: 390, lat: 815, lon: 341, z_l: 52, zi: 76)
Coordinates:
  * time                         (time) datetime64[ns] 3kB 1993-01-16T12:00:0...
  * lat                          (lat) float64 7kB 10.81 10.89 ... 80.63 80.72
  * lon                          (lon) float64 3kB 156.9 157.2 ... 254.7 255.0
  * z_l                          (z_l) float64 416B 2.5 7.5 ... 6.25e+03
  * zi                           (zi) float64 608B 0.0 2.0 ... 6.25e+03 6.5e+03
Data variables: (12/356)
    FRAZIL                       (time, lat, lon) float32 434MB ...
    LSNK                         (time, lat, lon) float32 434MB ...
    MIB                          (time, lat, lon) float32 434MB ...
    baccos                       (time, lat, lon) float32 434MB ...
    btm_co3_ion                  (time, lat, lon) float32 434MB ...
    btm_co3_sol_arag             (time, lat, lon) float32 434MB ...
    ...                           ...
    wc_vert_int_jprod_no3nitrif  (time, lat, lon) float32 434MB ...
    zmesoos                      (time, lat, lon) float32 434MB ...
    zoocos                       (time, lat, lon) float32 434MB ...
    wc_vert_int_npp              (time, lat, lon) float32 434MB ...
    zmicroos                     (time, lat, lon) float32 434MB ...
    wc_vert_int_nfix             (time, lat, lon) float32 434MB ...
Attributes: (12/28)
    NumFilesInSet:          1
    title:                  NEP10k_202507_physics_bgc
    grid_type:              regular
    grid_tile:              N/A
    NCO:                    netCDF Operators version 5.2.4 (Homepage = http:/...
    cefi_rel_path:          cefi_portal/northeast_pacific/full_domain/hindcas...
    ...                     ...
    associated_files:       areacello: 19930101.ocean_static.nc
    external_variables:     areacello
    cefi_ori_category:      ocean_cobalt_omip_sfc
    cefi_filename:          zoocos.nep.full.hcast.monthly.regrid.r20250912.19...
    cefi_variable:          zoocos
    cefi_ori_filename:      ocean_cobalt_omip_sfc.199301-202506.zoocos.nc